# Representation geometry: dual spaces, contrasts and categorical diagnostics

This download-free walkthrough uses a known synthetic readout and a tiny randomly initialized TransformerBridge. It demonstrates algebra and API contracts, not the geometry of trained language models. The metric is motivated by [Park, Choe and Veitch](https://arxiv.org/abs/2311.03658v2); model-level causal separability and paper reproduction are not established here.

Run all cells on CPU against the current TransformerLens installation. No pretrained weights, external tokenizer, learned artifact or HF token is needed.

In [1]:
import math
import sys
from pathlib import Path

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "pyproject.toml").is_file() and (candidate / "transformer_lens").is_dir():
        sys.path.insert(0, str(candidate))
        break

import torch
from transformer_lens.tools.analysis import RepresentationGeometry

torch.manual_seed(42)
dtype = torch.float64


## 1. Centered population covariance

The readout has layout `[d_model, d_vocab]`. Four shifted token vectors below have known covariance `R @ diag(4, 1) @ R.T`. Centering removes the arbitrary common token offset. Population normalization divides by the token count, not one less than that count.

In [2]:
rotation = torch.tensor([[0.6, -0.8], [0.8, 0.6]], dtype=dtype)
axes = torch.tensor([[1.0, -1.0, 0.0, 0.0], [0.0, 0.0, 1.0, -1.0]], dtype=dtype)
readout = math.sqrt(2) * rotation @ torch.diag(torch.tensor([2.0, 1.0], dtype=dtype)) @ axes
readout = readout + torch.tensor([[7.0], [-3.0]], dtype=dtype)
geometry = RepresentationGeometry(readout)
expected_covariance = rotation @ torch.diag(torch.tensor([4.0, 1.0], dtype=dtype)) @ rotation.T
torch.testing.assert_close(geometry.covariance, expected_covariance)
whitening = geometry.whiten_measurement(torch.eye(2, dtype=dtype))
torch.testing.assert_close(whitening @ geometry.covariance @ whitening.T, torch.eye(2, dtype=dtype))
assert geometry.diagnostics.measured_rank == 2
print("Known population covariance and whitening identity verified.")


Known population covariance and whitening identity verified.


## 2. Measurement and intervention vectors transform dually

For column vectors, measurement whitening uses `W = M^(-1/2)` and intervention whitening uses `S = M^(1/2)`. Their pairing is preserved. The following raw measurement vectors are deliberately constructed so their whitened coordinates are orthogonal; this is not evidence that a real concept pair is causally separable. Raw cosine is a comparison baseline, not universally invalid.

In [3]:
measurements = geometry.unwhiten_measurement(torch.eye(2, dtype=dtype))
a, b = measurements
raw_cosine = torch.nn.functional.cosine_similarity(a, b, dim=0)
metric_cosine = geometry.measurement_cosine(a, b)
assert abs(metric_cosine.item()) < 1e-12
assert abs(raw_cosine.item()) > 0.5
context = torch.tensor([2.0, -1.0], dtype=dtype)
pairing = geometry.whiten_intervention(context) @ geometry.whiten_measurement(a)
torch.testing.assert_close(pairing, context @ a)
torch.testing.assert_close(geometry.unwhiten_measurement(geometry.whiten_measurement(a)), a)
torch.testing.assert_close(
    geometry.unwhiten_intervention(geometry.whiten_intervention(context)), context
)
print(
    f"Constructed fixture: raw cosine {raw_cosine.item():.3f}, absolute metric cosine {abs(metric_cosine.item()):.3f}."
)


Constructed fixture: raw cosine 0.584, absolute metric cosine 0.000.


## 3. Oriented counterfactual pairs and dispersion

Pair `(lo, hi)` contributes `W_U[:, hi] - W_U[:, lo]`. The result is an unnormalized mean, with raw and whitened population dispersion. A single pair has dispersion zero, not a confidence guarantee. Derived intervention coordinates follow `M^(-1) @ measurement`; their alignment is an algebraic identity, not an independently observed intervention.

In [4]:
concept = geometry.concept_direction([(1, 0), (3, 2)], label="synthetic contrast")
reversed_concept = geometry.concept_direction([(0, 1), (2, 3)])
torch.testing.assert_close(reversed_concept.raw_measurement, -concept.raw_measurement)
torch.testing.assert_close(reversed_concept.whitened_dispersion, concept.whitened_dispersion)
torch.testing.assert_close(concept.whitened_derived_intervention, concept.whitened_measurement)
single = geometry.concept_direction([(1, 0)])
assert single.raw_dispersion.item() == 0.0
assert concept.intervention_kind == "metric-derived"
print(f"{concept.n_pairs} equally weighted pairs; intervention is {concept.intervention_kind}.")


2 equally weighted pairs; intervention is metric-derived.


## 4. Singular covariance and explicit ridge

A singular exact fit raises rather than silently pseudo-inverting or clamping eigenvalues. An explicit ridge changes the metric: the identity is `W @ (Sigma + epsilon I) @ W.T == I`, not identity covariance for the original whitened rows.

In [5]:
singular_readout = torch.ones(2, 4, dtype=dtype)
try:
    RepresentationGeometry(singular_readout)
except ValueError as error:
    assert "rank-deficient" in str(error)
else:
    raise AssertionError("An exact singular fit must be rejected.")
regularized = RepresentationGeometry(singular_readout, ridge=0.25)
ridge_whitening = regularized.whiten_measurement(torch.eye(2, dtype=dtype))
torch.testing.assert_close(
    ridge_whitening @ regularized.regularized_covariance @ ridge_whitening.T,
    torch.eye(2, dtype=dtype),
)
torch.testing.assert_close(
    ridge_whitening @ regularized.covariance @ ridge_whitening.T, torch.zeros(2, 2, dtype=dtype)
)
print("Singular exact fit rejected; explicit regularized identity verified.")


Singular exact fit rejected; explicit regularized identity verified.


## 5. Simplex is not the same question as regular simplex

The API accepts explicit vertices in a declared raw space. The examples below construct a regular triangle, a non-regular triangle and collinear points. They are controlled fixtures, not category representations estimated from language-model tokens. Degenerate vertices are retained; undefined angles are NaN with an explicit validity mask.

In [6]:
triangle = torch.tensor(
    [[1.0, 0.0], [-0.5, math.sqrt(3) / 2], [-0.5, -math.sqrt(3) / 2]], dtype=dtype
)
regular = geometry.categorical_geometry(
    geometry.unwhiten_measurement(triangle), space="measurement", labels=["a", "b", "c"]
)
scalene = torch.tensor([[0.0, 0.0], [2.0, 0.0], [0.0, 1.0]], dtype=dtype)
non_regular = geometry.categorical_geometry(
    geometry.unwhiten_measurement(scalene), space="measurement"
)
line = torch.tensor([[-1.0, 0.0], [0.0, 0.0], [1.0, 0.0]], dtype=dtype)
degenerate = geometry.categorical_geometry(geometry.unwhiten_measurement(line), space="measurement")
assert regular.is_simplex() and regular.is_regular_simplex()
assert non_regular.is_simplex() and not non_regular.is_regular_simplex()
assert not degenerate.is_simplex()
assert torch.isnan(degenerate.angles[~degenerate.angle_valid_mask]).all()
print(
    {
        "regular triangle": regular.is_regular_simplex(),
        "non-regular triangle is simplex": non_regular.is_simplex(),
        "collinear points are simplex": degenerate.is_simplex(),
    }
)


{'regular triangle': True, 'non-regular triangle is simplex': True, 'collinear points are simplex': False}


## 6. A tiny, local TransformerBridge with a real tokenizer

This model has random weights and only four residual features. It validates integration, not semantic gender representations. The local tokenizer deliberately inserts BOS/EOS by default; geometry string encoding disables them and requires one known token. The basis is the input to the raw linear vocabulary head after final normalization, not a pre-normalization residual hook.

In [7]:
from tokenizers import Tokenizer
from tokenizers.models import WordLevel
from tokenizers.pre_tokenizers import Whitespace
from tokenizers.processors import TemplateProcessing
from transformers import GPT2Config, GPT2LMHeadModel, PreTrainedTokenizerFast
from transformer_lens.model_bridge.sources import build_bridge_from_module

vocab = {
    "<unk>": 0,
    "<bos>": 1,
    "<eos>": 2,
    "<pad>": 3,
    "king": 4,
    "queen": 5,
    "man": 6,
    "woman": 7,
}
backend = Tokenizer(WordLevel(vocab, unk_token="<unk>"))
backend.pre_tokenizer = Whitespace()
backend.post_processor = TemplateProcessing(
    single="<bos> $A <eos>", special_tokens=[("<bos>", 1), ("<eos>", 2)]
)
tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=backend,
    unk_token="<unk>",
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
)
config = GPT2Config(
    vocab_size=8,
    n_embd=4,
    n_layer=1,
    n_head=2,
    n_positions=16,
    resid_pdrop=0.0,
    embd_pdrop=0.0,
    attn_pdrop=0.0,
    bos_token_id=1,
    eos_token_id=2,
    pad_token_id=3,
)
config._attn_implementation = "eager"
model = GPT2LMHeadModel(config).eval()
bridge = build_bridge_from_module(
    model,
    "GPT2LMHeadModel",
    hf_config=config,
    tokenizer=tokenizer,
    device="cpu",
    model_name="tiny-random-geometry",
)
state_before = {name: value.detach().clone() for name, value in bridge.state_dict().items()}
bridge_geometry = RepresentationGeometry.from_bridge(bridge, compute_dtype=torch.float64)
strings = bridge_geometry.concept_direction(
    [("king", "queen"), ("man", "woman")], label="toy contrast"
)
ids = bridge_geometry.concept_direction([(4, 5), (6, 7)])
torch.testing.assert_close(strings.raw_measurement, ids.raw_measurement)
assert strings.pairs == ((4, 5), (6, 7))
assert tokenizer.encode("king", add_special_tokens=True) == [1, 4, 2]
for name, value in bridge.state_dict().items():
    torch.testing.assert_close(value, state_before[name], rtol=0, atol=0)
print(
    {
        "readout shape": list(bridge_geometry.diagnostics.input_shape),
        "basis": bridge_geometry.basis.input_location,
    }
)


{'readout shape': [4, 8], 'basis': 'post-final-normalization'}


In [8]:
captured = []


def observe_readout_input(value, hook):
    captured.append(value.detach().clone())


with torch.no_grad():
    logits = bridge.run_with_hooks(
        torch.tensor([[4, 6]]), fwd_hooks=[("unembed.hook_in", observe_readout_input)]
    )
normalized_context = captured[0]
pairing = (
    bridge_geometry.whiten_intervention(normalized_context) * strings.whitened_measurement
).sum(-1)
mean_logit_difference = ((logits[..., 5] - logits[..., 4]) + (logits[..., 7] - logits[..., 6])) / 2
bias_difference = ((bridge.b_U[5] - bridge.b_U[4]) + (bridge.b_U[7] - bridge.b_U[6])) / 2
torch.testing.assert_close(
    pairing, (mean_logit_difference - bias_difference).to(dtype), atol=1e-6, rtol=1e-5
)
print("Post-normalization pairing matches the bias-adjusted average logit difference.")


Post-normalization pairing matches the bias-adjusted average logit difference.


## Research boundaries

- These tests check analytic identities and tiny-model integration, not learned semantic geometry.
- Pair quality and independent interventions need controlled, held-out model experiments. Raw cosine can remain a useful baseline.
- Pre-normalization or earlier-layer steering needs a valid contextual basis mapping; dimensions alone do not establish compatibility.
- Dense whitening changes coordinate sparsity. Probe coefficients need preprocessing/support reconstruction before comparisons.
- Report model/tokenizer provenance, processing state, dtype, ridge, rank thresholds and contrast construction. Dense covariance/eigendecomposition can be expensive despite requiring no model training.

See the [representation geometry guide](https://TransformerLensOrg.github.io/TransformerLens/content/representation_geometry.html) for the full support, precision, resource and interpretation contracts.